# Human Interaction

**Module:** 14 — AI Orchestration

Human-in-the-loop patterns, UX, and escalation for orchestrated AI.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain why HITL exists in AI workflows
- Implement approval / edit / reject loops
- Design UX patterns that respect latency and trust


## Why Humans in the Loop?

### Definition
Humans provide judgment, accountability, and authority that models and tools should not unilaterally exercise for high-impact actions.

### Why it matters
Refunds, legal emails, medical advice, production deploys — wrong automation is costly. HITL is a control-plane feature.

### How it works
Workflow pauses in `waiting_human`, notifies reviewers, accepts decisions as events, resumes with audit trail.

### Intuition
Co-pilot with a stick shifter for dangerous roads.

### Pitfalls
- HITL on everything (ops burnout)
- HITL on nothing critical
- No SLA for reviewers

### When to use
Irreversible, ambiguous, regulated, or brand-sensitive actions.


| Mode | Human role | Example |
|------|------------|---------|
| Approve/reject | Gate | Send customer email |
| Edit then approve | Co-author | Marketing copy |
| Provide data | Form fill | Missing account id |
| Take over | Escalation | Angry VIP |
| Audit async | Sample review | Score 2% of auto replies |

```mermaid
sequenceDiagram
  participant W as Workflow
  participant H as Human
  W->>W: draft action
  W->>H: request review
  H->>W: approve/edit/reject
  W->>W: resume / compensate
```


In [ ]:
# Demo 1: approval gate
from dataclasses import dataclass, field

@dataclass
class HITLInbox:
    pending: dict = field(default_factory=dict)
    def request(self, run_id, payload):
        self.pending[run_id] = {"payload": payload, "decision": None}
        return {"status": "waiting_human", "run_id": run_id}
    def decide(self, run_id, decision, edited=None):
        item = self.pending[run_id]
        item["decision"] = decision
        if edited is not None:
            item["payload"] = edited
        return item

inbox = HITLInbox()
print(inbox.request("r1", {"email_body": "Sorry for the outage."}))
print(inbox.decide("r1", "approve"))
print(inbox.decide("r9", "edit", {"email_body": "Edited apology."}) if "r9" in inbox.pending else "no r9")


## UX Patterns

### Definition
Interfaces that make review fast: diffs, risk badges, suggested actions, keyboard approve, mobile-friendly queues.

### Why it matters
If review is painful, humans rubber-stamp or abandon the queue — both defeat the purpose.

### How it works
Show: why paused, what will happen, evidence/citations, blast radius, timeout policy. Capture structured decisions.

### Intuition
Good HITL feels like code review, not archaeology.

### Pitfalls
- Wall of unstructured text
- No undo messaging
- Approving without seeing tool args

### When to use
Any product with a review queue.


In [ ]:
# Demo 2: render a review card (text UI)
def review_card(run):
    return f"""
## Review needed — {run['run_id']}
Risk: {run['risk']}
Action: {run['action']}
Args: {run['args']}
Evidence: {run.get('evidence', [])}
SLA: {run.get('sla_minutes', 60)} minutes
Commands: /approve  /reject  /edit
""".strip()

print(review_card({
    "run_id": "r1",
    "risk": "high",
    "action": "issue_refund",
    "args": {"amount": 250, "order_id": "O-9"},
    "evidence": ["policy:refunds_30d", "crm:vip"],
}))


In [ ]:
# Demo 3: escalation policy
def needs_human(action: str, amount: float, confidence: float) -> bool:
    if action in {"issue_refund", "delete_account", "send_legal"}:
        if amount is not None and amount >= 100:
            return True
    if confidence < 0.6:
        return True
    return False

tests = [
    ("issue_refund", 250, 0.95),
    ("search_kb", 0, 0.4),
    ("send_faq", 0, 0.9),
]
for t in tests:
    print(t, needs_human(*t))


In [ ]:
# Demo 4: timeout then escalate
from datetime import datetime, timedelta, timezone

def review_status(requested_at, now, sla_minutes=60):
    if now - requested_at <= timedelta(minutes=sla_minutes):
        return "waiting_human"
    return "escalate_to_secondary_queue"

t0 = datetime(2026, 8, 1, 12, 0, tzinfo=timezone.utc)
print(review_status(t0, t0 + timedelta(minutes=30)))
print(review_status(t0, t0 + timedelta(minutes=90)))


In [ ]:
# Demo 5: audit log entry shape
import json, os
audit_event = {
    "run_id": "r1",
    "actor": "user:approver_7",
    "decision": "approve",
    "action": "issue_refund",
    "before": {"amount": 250},
    "after": {"amount": 250},
    "ts": "2026-08-01T12:00:00Z",
}
print(json.dumps(audit_event, indent=2))
print("export to SIEM; API key for SIEM:", os.getenv("SIEM_TOKEN", "YOUR_SIEM_TOKEN")[:8] + "...")


### Try it yourself — HITL

1. Implement reject → compensation (`cancel_draft`).
2. Add dual-control: two distinct approvers for amount >= 1000.
3. Sketch a mobile review UI wireframe in ASCII.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `HITL` | Human in the loop |
| `escalation` | Route to higher-authority queue |
| `rubber stamping` | Approving without real review |
| `dual control` | Two-person approval |


## Reviewer Fatigue Model

If you send too many low-risk items to humans, approval quality collapses.

| Signal | Auto | HITL |
|--------|------|------|
| FAQ draft, conf>0.85 | ✓ | sample 2% |
| Refund <$50 | ✓ | audit async |
| Refund ≥$100 | | ✓ |
| Legal/threat language | | ✓ dual control |

```
precision of automation ↑  when thresholds tuned on labeled tickets
```


In [ ]:
# Dual control
class DualControl:
    def __init__(self):
        self.approvals = {}
    def approve(self, run_id, actor):
        self.approvals.setdefault(run_id, set()).add(actor)
        return len(self.approvals[run_id]) >= 2

dc = DualControl()
print(dc.approve("r1", "alice"))
print(dc.approve("r1", "alice"))  # same actor
print(dc.approve("r1", "bob"))


In [ ]:
# Queue priority
tickets = [
    {"id": 1, "vip": True, "risk": "high", "age_m": 10},
    {"id": 2, "vip": False, "risk": "low", "age_m": 120},
    {"id": 3, "vip": False, "risk": "high", "age_m": 30},
]

def priority(t):
    return (not t["vip"], 0 if t["risk"] == "high" else 1, -t["age_m"])

print([t["id"] for t in sorted(tickets, key=priority)])


### Try it yourself — HITL deepen

1. Add expiry: if not dual-approved in 60m, escalate.
2. Design email vs Slack notification payloads for review cards.


## Key Takeaways

- HITL is a first-class workflow state
- UX quality determines safety outcomes
- Audit every decision
- Tune thresholds to avoid reviewer fatigue
